# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [70]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa", "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa", "Ecoli_example_sequences.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 14
heterologous_genes  = ["hpaBC","ddc"]
# heterologous_genes  = ["ddc"]#,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(background_genes, heterologous_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))


## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)
# Are mRNA gene names in E_coli gene 


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);

# Integration
# PERFORMANCE FIX: Use Diagonal mass matrix instead of dense matrix
# For 100 sequences with ~22k states, dense matrix would be ~4GB and cause O(n³) scaling
n_all = length(y0_all)
mass_diag_all = ones(Float64, n_all)
mass_diag_all[conserved] .= 0.0
M = Diagonal(mass_diag_all)  # Much more memory-efficient: O(n) instead of O(n²)

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p       = generate_p(local_tags_list)

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p); ## Create an ODE problem

# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (distributed, test set)...")

sol     = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6); ## Solve ODE system

println("Integration completed!")
# save_results(sol, local_tags_list, save_data)


🔧 Julia started with 8 threads


Number of sequences: 16

⏳ Integrating ODE system (distributed, test set)...


LoadError: TaskFailedException

[91m    nested task error: [39mBoundsError: attempt to access 78-element Vector{Float64} at index [106]
    Stacktrace:
     [1] [0m[1mthrow_boundserror[22m[0m[1m([22m[90mA[39m::[0mVector[90m{Float64}[39m, [90mI[39m::[0mTuple[90m{Int64}[39m[0m[1m)[22m
    [90m   @[39m [90mBase[39m [90m.\[39m[90m[4messentials.jl:14[24m[39m
     [2] [0m[1mgetindex[22m
    [90m   @[39m [90m.\[39m[90m[4messentials.jl:916[24m[39m[90m [inlined][39m
     [3] [0m[1mrhs_sequence![22m[0m[1m([22m[90mdu[39m::[0mVector[90m{Float64}[39m, [90mu[39m::[0mVector[90m{Float64}[39m, [90mp[39m::[0mMain.txtl_types.data_struct_integration[0m[1m)[22m
    [90m   @[39m [36mMain[39m [90mc:\Users\ac135292\TXTLv2\src\[39m[90m[4mrhs_sequence.jl:158[24m[39m
     [4] [0m[1mODE![22m
    [90m   @[39m [90mc:\Users\ac135292\TXTLv2\src\[39m[90m[4mfunctions.jl:1314[24m[39m[90m [inlined][39m
     [5] [0m[1mmacro expansio

In [66]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
# u = similar(y0_all)
t = tspan;

# @benchmark ODE_parallel_distributed!(du, u, p, t)


In [67]:

# Vectors and numbers 
@unpack y_codons_lengths, offsets, n_metab, n_seq, du_metab_all, du_codons, y_codons, results, du_seq, du_metab = p.vectors

# Generate parts of u that need to be assembled specific to the sequence   
@views u_metab =  u[1:n_metab]				                            # Save all pool species from u in a vector 

i = 1

local_tags = p.local_tags_list[i]		                            # Extraction of local tags
idx = n_metab + offsets[i]                                          # Create index to access u on the sequence specific indexes
len = y_codons_lengths[i]            

# thread-safe local views
u_codons = @view u[idx+1 : idx+len]

# assemble without allocation
u_seq = similar(u, n_metab + len)
@views u_seq[1:n_metab] .= u_metab
@views u_seq[n_metab+1:end] .= u_codons

du_seq_local = du_seq[i]					                        # Initialization of sequence specific rates vector

du = du_seq_local
fill!(du, 0.0)
u = u_seq
p = local_tags

# @benchmark rhs_metabolites!(du, u, p)
rhs_metabolites!(du, u, p)


In [68]:
@benchmark rhs_sequence!(du, u, p)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  38.300 μs …  22.820 ms  ┊ GC (min … max): 0.00% … 99.56%
 Time  (median):     44.800 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   50.894 μs ± 228.261 μs  ┊ GC (mean ± σ):  4.46% ±  1.00%

  ▇█▆▆▇██▇▆▄▃▃▂▁▁                         ▁▂▂▂▁▂               ▂
  ████████████████▇▇▅▅▄▆▆▅▆▄▇▅▅▅▅▅▄▂▃▅▄▅▇█████████▇▇▇▇▆▆▇▇▇▅▅▆ █
  38.3 μs       Histogram: log(frequency) by time       109 μs <

 Memory estimate: 39.56 KiB, allocs estimate: 46.

In [69]:
function rhs_sequence_test!(du, u, p)

    # du = zeros(Float64, length(u))
    tags = p
    # Unpack parameters 
    @unpack µ, k_proc, k_cat_Pol, C_RNP, V_TCmax, K_M_D, k_cat_E, C_RNE, V_degmax, K_M_deg, k_D, n_DNA_t, n_DNA_s = tags.p_seq.TX
    @unpack k_TLEmaxj, KM_T3j, KM_EFGGTP = tags.p_seq.EL
    @unpack k_TE, KM_RK, KM_GTP, KM_RF3GTP, k_RF3_diss_GDP, k_RF1_diss_GTP, k_GTP_hydr, k_RF1_ass = tags.p_seq.TER
    @unpack k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss = tags.p_seq.RF3
    @unpack A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split = tags.p_seq.PTC 
    
    # Unpack constant variables 
    ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, 
    Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, val, Trp, Tyr, C_D = tags.init_c.constant_vars 

    # Unpack dynamic variables 
    S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, 
    tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, 
    tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, 
    r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, S70, rna, PRT, PTC = u 


    # Define RIB1
    
    fill!(tags.rate_RIB, 0.0)

    @views c_RIB	= u[tags.idx_first_codon:end]     

    @views total_RNA   = tags.RNA + sum(c_RIB[1:12])

    fill!(tags.q_R_elo, 1.0)
        
    @views for i in 2:(tags.len_sequence - 13)

        if total_RNA-sum(c_RIB[i:i+12]) == 0

            tags.q_R_elo[i] 	= 0;
        else
            tags.q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
        
        end
    end 


    @views for i in (tags.len_sequence-13) : (tags.len_sequence-1)
        if total_RNA - sum(c_RIB[i : end]) == 0
            tags.q_R_elo[i] = 0
        else 
            tags.q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
        end
    end

    # dynamic efficiency factor 
    if tags.flags.dynamic_efficiency == true

        efac = dynEffficiencyCalculation(tags, EGT, u);

    end
        

    # V_EL
    C_vect = reshape(u[1:tags.num_species], 1, tags.num_species)  # Directly assign
    c_Codon = tags.t3_Codon_struc * C_vect'

    V_TL_EL	= zeros(tags.len_sequence-1);

    EGT_inv = inv(EGT)

    #   #   #   #   #   #   14 µs   #   #   #   #   #       #




    @inbounds for i in 1:tags.len_sequence-2

        cod_next = tags.species_and_sequences.Seq[i+1]
        cod_curr = tags.species_and_sequences.Seq[i]

        j = tags.species_and_sequences.cod_idx[cod_next]    # O(1) Lookup
        k = tags.species_and_sequences.cod_idx[cod_curr]    # O(1) Lookup

        cc = c_Codon[j]

        if cc == 0
            V_TL_EL[i+1] = 0
        else
            eff = efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i])

            denom = 1 + KM_T3j/cc + KM_EFGGTP*EGT_inv

            V_TL_EL[i+1] =
                tags.q_R_elo[i] * k_TLEmaxj * c_RIB[i] * eff / denom
        end
    end

    #   #   #   #   #   #   #   34.700 μs   #   #   #   #   #   #   #   #   #

    V_TL_EL_sum = sum(V_TL_EL)
    
    c_codon_seq = c_Codon[tags.matrix_N]                    # 35.400 μs     # Projects the concentration of every codon (of the 61 codon) to every codon in the sequence
    
    
    # N= C_vect' * (1 ./c_codon_seq)'                         # 102.700 μs    # Dividing actual t3 complex concentration by c_Codon_Codonsequenz produces array N 
    #                                                                         # All species are divided by codon concentrations
    
    # fill!(tags.M, 0.0)                                      # 109.300 μs
    
    # tags.M .= tags.alpha_struc .* N                         # 128.300 μs    # Alpha struc is an array which combines the t3-complexes with the correct codon from the codon sequence.
    #                                                                         # Alpha struc: Which t3 complex binds which codon? Species x Codons        
    #                                                                         # M is a sparse matrix with entries at the same positions like alpha_struc
    

    # K_T3 = tags.M*V_TL_EL                                   # 139.000 μs    # Rate of T3 consumption
    
    # M_tRNA = @view tags.M[:,2:end-1]                        # 139.300 μs 


    # V_TL_EL_subset = @view V_TL_EL[3:end]                   # 141.000 μs
    
    # K_tRNA = M_tRNA * V_TL_EL_subset                        # unloaded tRNA released by T3 complex consumptions

    #   #   #   #   #   149.400 μs  #   #   #   #   #   #   #   #

    K_T3, K_tRNA, N_one, N_two = matrix_free_KT3(tags, V_TL_EL, c_codon_seq, C_vect)

    #   #   #   #   #   152.100 μs  #   #   #   #   #   #   #   #

end

@benchmark rhs_sequence_test!(du, u, p)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  37.000 μs …  18.039 ms  ┊ GC (min … max): 0.00% … 99.64%
 Time  (median):     41.700 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   45.769 μs ± 180.377 μs  ┊ GC (mean ± σ):  3.93% ±  1.00%

  ██▆▅▇▇▇▇▆▄▃▃▂▁▁                                    ▁▁▂▂▁▁▁   ▃
  ████████████████▇▇▇▅▇▅▆▆▆▆▄▅▆▆▅▆▁▅▄▄▃▄▁▄▃▄▁▄▃▁▅▄▆▇████████▇▇ █
  37 μs         Histogram: log(frequency) by time      93.1 μs <

 Memory estimate: 31.14 KiB, allocs estimate: 36.

In [ ]:

      
    ## translation termination
    te = tags.len_sequence-1 # Number of last coding codon
    # ke = findfirst(x -> x == tags["Seq1"]["Codon"][te], tags["cod"])

    k_RF1_diss_GTP = k_RF1_diss_GTP#*tags.scale.termination

    efficiency_term = 1;
    k_ter_RF1 = 1 / ( 1/(k_RF1_ass*RFt) + 1/(k_RF1_diss_GTP) );
    k_ter_RF3 = 1 / ( 1/(k_RF1_ass*RGT) + 1/(k_RF3_diss_GDP) + 1/(k_GTP_hydr) );

    if k_ter_RF1 < k_ter_RF3
        k_ter = k_ter_RF1;
    else
        k_ter = k_ter_RF3;
    end


    V_TE_max = k_ter * RFt   # RFt is RF1/2 total
    V_TL_TE = V_TE_max * efficiency_term / (1 + KM_RK/ c_RIB[te] + KM_RF3GTP/RGT + KM_RK * KM_RF3GTP /(c_RIB[te] * RGT))
        
    if c_RIB[te] == 0 || GTP == 0
        V_TL_TE=0;
    end

    V_RF3_GDP_ass  = k_RF3_GDP_ass  * RF3 * GDP;
    V_RF3_GTP_ass  = k_RF3_GTP_ass  * RF3 * GTP;
    V_RF3_GDP_diss = k_RF3_GDP_diss * RGD;
    V_RF3_GTP_diss = k_RF3_GTP_diss * RGT 

    ## post termination complex recycling
    t_rec = A1 + A2 * 1/RRF + A3 * EGT/RRF + A4 * 1/EGT + A5 * 1/(RRF*EGT); # time/s for recycling according to Borg et al. 2016
    k_rec = 1/t_rec; # rate/s^-1 for recycling according to Borg et al. 2016
    k_rec_min = k_rec*60; # rate/min^-1 for recycling according to Borg et al. 2016
    f_s = k_cat_K_M_G1/(k_RRF * RRF)*(q_RRF/k_cat_K_M_G2*one_q_G2_k_split + EGT)+ one_q_G2_k_split; #EGT hydrolised per splitting event according to Borg et al 2016

    # translation termination - calculation of du 

    # @inbounds for i in 1:tags.num_species
    #     du[i] += tags.alpha_struc[i, end] * V_TL_TE

    #     if tags.alpha_struc[i, end] * V_TL_TE != 0
    #     # println(tags.alpha_struc[i, end] * V_TL_TE)
    #     println("alpha ", tags.alpha_struc[i, end])
    #     println("V ",V_TL_TE)
    #     end
    # end

    du[tags.alpha_one[end]] += N_one[tags.alpha_one[end]] * V_TL_TE

    if tags.alpha_two[end] != 0
        du[tags.alpha_two[end]] += N_two[tags.alpha_two[end]] * V_TL_TE
    end


In [62]:
N_one[104]

1.0

In [52]:
tags.alpha_struc[:,end]

133-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [54]:
for i in tags.alpha_struc[:,end]
println(i)
end

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
1.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [47]:
tags.alpha_one[end]

104

In [43]:
for i in du println(i) end

-0.0008558200074730848
-0.0008558742010882442
-6.966242273244612e-9
-1.974475495537982e-9
5.41936151274942e-8
-5.672677616530564e-8
5.586855422734516e-10
-4.4330779047641045e-9
-6.35848929064764e-10
7.043402075446702e-9
0.0
-2.7569058147491887e-12
0.5190772754722275
-0.5190772754140198
9.840645925907054e-8
0.0
0.0
0.0
-7.0434021073222455e-9
-0.03252343536719404
-0.020541173594720893
-0.007274960312759875
-0.030811669440523767
-0.026960209529765366
-0.014549925162412661
-0.006419120133982835
-0.030383734454335957
-0.013266137204897445
-0.023536667224193595
-0.015405784741090044
-0.03125728273899049
-0.008130855695764641
-0.00554561260970758
-0.0033765754876510777
-0.003470482440244984
-0.015405793574484437
-0.01925724907492019
-0.006901893722836238
-0.004680272974785263
-0.0106707408499016
-0.007274980678338026
-0.016261744929716878
-0.026532255284308824
-0.00385149768416269
-3.31159768687086e-21
-1.9847887852500513e-21
-0.009631083096996633
-0.0013569773170170293
-0.011982317060431043


In [ ]:
    @inbounds for i in 1:tags.num_species
        du[i] += tags.alpha_one[end] * V_TL_TE
    end

In [ ]:
du[alpha_one[end]] += tags.alpha_one[end] * V_TL_TE
du[alpha_two[end]] += tags.alpha_two[end] * V_TL_TE

104

In [11]:
V_TL_TE

0.0008558845177561139

In [15]:
i =10
tags.alpha_struc[:, end]

133-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [16]:
for i in tags.alpha_struc[:, end]
    println(i)
end

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
1.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [ ]:

    tags.rate_RIB[tags.len_sequence-1] -= V_TL_TE


    du[tags.idx.PTC] += V_TL_TE - k_rec_min * PTC;	            # PTC
    du[tags.idx.S30] += k_rec_min * PTC;                         # rate of 30S release due to recycling
    du[tags.idx.S50] +=  k_rec_min * PTC;                        # rate of 50S release due to recycling
    du[tags.idx.PRT] = V_TL_TE;                                  # rate of PRT (Protein)
    du[tags.idx.RGT] = -V_TL_TE -V_RF3_GTP_diss +V_RF3_GTP_ass;  # rate of RGT
    du[tags.idx.RGD] =  V_TL_TE -V_RF3_GDP_diss +V_RF3_GDP_ass;  # rate of RGD
    du[tags.idx.RF3] =          +V_RF3_GDP_diss +V_RF3_GTP_diss -V_RF3_GTP_ass -V_RF3_GDP_ass; # rate of RF3


    # translation elongation - calculation of du 



    # concentration of the ribosome at the specific codon
    tags.rate_RIB[1] = - V_TL_EL[2];  

    for i in 2: tags.len_sequence-2
        tags.rate_RIB[i] = V_TL_EL[i] - V_TL_EL[i+1];	
    end

    tags.rate_RIB[tags.len_sequence-1] = tags.rate_RIB[tags.len_sequence-1] + V_TL_EL[end];

    # concentration of the unloaded tRNA
    du[tags.idx.ta1:tags.idx.ty1] .+= K_tRNA[end-tags.species_and_sequences.no_of_tRNAs+1:end]

    # concentration of the ternary-complex
    du[1: tags.num_species] .+= - K_T3;
    du[tags.idx.EGT]  += - V_TL_EL_sum - f_s * k_rec_min * PTC;	# Rate of EFG_GTP 
    du[tags.idx.EGD]  += + V_TL_EL_sum + f_s * k_rec_min * PTC;	# Rate of EFG_GDP

    du[tags.idx.rna] += - V_TL_EL[12];



    # save RIB in du
    du[tags.idx_first_codon:end] .+= tags.rate_RIB;

    return du
end

## Calculate K_T3

### Use alpha vectors instead of alpha struc matrix

In [18]:
# function matrix_free_KT3(tags, V_TL_EL, c_codon_seq, C_vect)
    
    ### Division of tRNA concentration by codon concentration # ---------------------------------------------------------------------

    N_one = zeros(698)
    N_two = zeros(698)
    # N_three = zeros(698)

    counter = 0
    for (index, c_codon) in zip(alpha_one, c_codon_seq)
        counter += 1
        N_one[counter] = C_vect[index] / c_codon
    end

    counter = 0
    for (index, c_codon) in zip(alpha_two, c_codon_seq)
        if index != 0.0
            counter += 1
            N_two[counter] = C_vect[index] / c_codon
        elseif index ==0.0
            counter += 1
            N_two[counter] = 0.0
        end
    end

    K_T3 = zeros(133) # ---------------------------------------------------------------------

    for (codon_position, tRNA_position) in enumerate(alpha_one) 
        K_T3[tRNA_position] += N_one[codon_position]*V_TL_EL[codon_position]
    end

    for (codon_position, tRNA_position) in enumerate(alpha_two)
        if tRNA_position != 0 
            K_T3[tRNA_position] += N_two[codon_position]*V_TL_EL[codon_position]
        end
    end

    K_tRNA = zeros(133) # ---------------------------------------------------------------------

    V_TL_EL_subset  = @view V_TL_EL[3:end] 
    N_one_subset    = @view N_one[2:end-1]
    N_two_subset    = @view N_two[2:end-1]
    alpha_one_subset= @view alpha_one[2:end-1]
    alpha_two_subset= @view alpha_two[2:end-1]

    for (codon_position, tRNA_position) in enumerate(alpha_one_subset) 
        K_tRNA[tRNA_position] += N_one_subset[codon_position]*V_TL_EL_subset[codon_position]
    end

    for (codon_position, tRNA_position) in enumerate(alpha_two_subset)
        if tRNA_position != 0 
            K_tRNA[tRNA_position] += N_two_subset[codon_position]*V_TL_EL_subset[codon_position]
        end
    end

#     return K_T3, K_tRNA

# end 


In [17]:
    alpha_one = zeros(Int, 698) # ---------------------------------------------------------------------
    alpha_two = zeros(Int, 698)
    alpha_three = zeros(Int, 698)
    num_col = 0

    for column in eachcol(tags.alpha_struc)
        num_col += 1
        num_element = 0.0

        for (index, element) in enumerate(column)
            if element !=0 && num_element == 0.0
                alpha_one[num_col] = index
                num_element += 1.0

            elseif element !=0 && num_element == 1.0
                alpha_two[num_col] = index
                num_element += 1.0

            elseif element !=0 && num_element == 2.0
                alpha_three[num_col] = index
                num_element += 1.0
            end
        end
    end

In [15]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6)

BenchmarkTools.Trial: 2 samples with 1 evaluation per sample.
 Range (min … max):  2.718 s …   2.808 s  ┊ GC (min … max): 11.50% … 12.45%
 Time  (median):     2.763 s              ┊ GC (median):    11.98%
 Time  (mean ± σ):   2.763 s ± 64.101 ms  ┊ GC (mean ± σ):  11.98% ±  0.67%

  █                                                       █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.72 s         Histogram: frequency by time        2.81 s <

 Memory estimate: 5.75 GiB, allocs estimate: 1179107.